# NB08. Rejection Cutoff Sensitivity

**목적**: AdvTop-k가 거절 비율 변화(5%~50%)에 안정적인지 확인. 실무에서 기관마다 거절 기준이 다르므로, 지표가 cutoff에 과도하게 민감하면 실용성이 떨어진다.

**Dependencies**: NB01 (datasets), NB02 (bb_shap, bb_prob), NB03 (surrogate contribs)

**핵심 질문**: 거절 비율을 5%→50%로 변화시킬 때 AdvTop-k가 얼마나 변동하는가?

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
from pathlib import Path

from decentra.metrics.attribution import advtopk

NB02_DIR = Path('../outputs/NB02')
NB03_DIR = Path('../outputs/NB03')
OUT_DIR = Path('../outputs/NB08')
OUT_DIR.mkdir(parents=True, exist_ok=True)

REJECT_PCTS = [95, 90, 80, 70, 50]  # percentile (top 5%, 10%, 20%, 30%, 50%)
METHODS = ['Tree-d1', 'EBM', 'Ridge', 'OptBin+Ridge']

all_results = {}

for data_name in ['GMSC', 'HC']:
    bb_shap = np.load(NB02_DIR / f'bb_shap_{data_name}.npy')
    bb_prob = np.load(NB02_DIR / f'bb_prob_{data_name}.npy')
    
    print(f'\n{"="*60}')
    print(f'  {data_name}: Cutoff Sensitivity')
    print(f'{"="*60}')
    print(f'{"Rej%":>5s}', end='')
    for m in METHODS:
        print(f'  {m:>14s}', end='')
    print()
    
    all_results[data_name] = {}
    
    for pctl in REJECT_PCTS:
        rej_pct = 100 - pctl
        cutoff = np.percentile(bb_prob, pctl)
        reject = bb_prob >= cutoff
        
        row = {'reject_pct': rej_pct}
        print(f'{rej_pct:>4d}%', end='')
        
        for method in METHODS:
            contribs = np.load(NB03_DIR / f'contribs_{method}_{data_name}.npy')
            at4 = advtopk(bb_shap, contribs, reject, 4, bb_sign=1, surr_sign=-1)
            at1 = advtopk(bb_shap, contribs, reject, 1, bb_sign=1, surr_sign=-1)
            row[f'{method}_AT4'] = round(at4, 4)
            row[f'{method}_AT1'] = round(at1, 4)
            print(f'  {at4:>6.3f}/{at1:.3f}', end='')
        
        print()
        all_results[data_name][f'top{rej_pct}pct'] = row

with open(OUT_DIR / 'cutoff_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved to {OUT_DIR}/')


  GMSC: Cutoff Sensitivity
 Rej%         Tree-d1             EBM           Ridge    OptBin+Ridge
   5%   0.933/0.821   0.929/0.852   0.546/0.315   0.906/0.820
  10%   0.930/0.808   0.927/0.842

   0.546/0.305   0.878/0.817
  20%   0.928/0.882   0.919/0.901

   0.550/0.272   0.859/0.873
  30%   0.931/0.911

   0.924/0.925   0.551/0.258

   0.862/0.889
  50%   0.934/0.908

...

   0.584/0.814
  50%

   0.799/0.892

   0.823/0.749

   0.575/0.000

   0.567/0.705

Saved to ..\outputs\NB08/
